In [2]:
import pandas as pd
import numpy as np

#Preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.feature_selection import SequentialFeatureSelector

#Model Machine Learning
from sklearn.tree import DecisionTreeRegressor
from catboost import CatBoostRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,mean_absolute_percentage_error,r2_score
import warnings
warnings.filterwarnings('ignore',category=UserWarning)

## 1. CatBoost Regression

In [3]:
df = pd.read_csv('../0.dataset/dataset_California_HousePrice_clean.csv')
df_x = df.drop(columns='median_house_value')
df_y = df['median_house_value']

In [4]:
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

In [5]:
feature_numerik = X_train.select_dtypes(include=np.number).columns
feature_categori = X_train.select_dtypes(include='object').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('categori_encoding',OneHotEncoder(drop='first', handle_unknown='ignore'),feature_categori)
    ],
    remainder='passthrough' 
)

In [6]:
selector_model = DecisionTreeRegressor(max_depth=2,random_state=42)
model_CatBoostRegressor = Pipeline([
    ('preprocessing',preprocessor),
    ('feature_selection',SequentialFeatureSelector(estimator=selector_model,direction='forward')),
    ('model_regression',CatBoostRegressor(random_state=42))
])

In [7]:
params={
    'feature_selection__n_features_to_select': [4, 5], # hindari 'auto' jika memicu terlalu banyak fitur
    'model_regression__iterations': [100, 250, 500],
    'model_regression__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model_regression__depth': [4, 6, 8],                       # Nilai default CatBoost adalah 6
    'model_regression__l2_leaf_reg': [1, 3, 5, 10],
    'model_regression__random_strength': [0, 1],
    'model_regression__bagging_temperature': [0.0, 1.0]
}
random_search = RandomizedSearchCV(estimator=model_CatBoostRegressor,param_distributions=params,n_iter=5,cv=5,scoring='r2',n_jobs=-1,verbose=1)
random_search.fit(X_train,y_train)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
0:	learn: 112390.7902912	total: 117ms	remaining: 11.6s
1:	learn: 109380.8068827	total: 120ms	remaining: 5.86s
2:	learn: 106611.7962901	total: 122ms	remaining: 3.94s
3:	learn: 104178.6394491	total: 124ms	remaining: 2.98s
4:	learn: 101742.2431204	total: 127ms	remaining: 2.41s
5:	learn: 99498.2778636	total: 129ms	remaining: 2.02s
6:	learn: 97412.7073891	total: 131ms	remaining: 1.74s
7:	learn: 95480.0882248	total: 134ms	remaining: 1.53s
8:	learn: 93672.5813227	total: 136ms	remaining: 1.37s
9:	learn: 92004.5330049	total: 138ms	remaining: 1.24s
10:	learn: 90484.4614197	total: 140ms	remaining: 1.14s
11:	learn: 89066.4215007	total: 143ms	remaining: 1.05s
12:	learn: 87785.5960174	total: 145ms	remaining: 972ms
13:	learn: 86580.3832230	total: 148ms	remaining: 906ms
14:	learn: 85503.2037210	total: 150ms	remaining: 849ms
15:	learn: 84472.2590559	total: 152ms	remaining: 799ms
16:	learn: 83522.5911647	total: 155ms	remaining: 757ms
17:	learn:

,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'feature_selection__n_features_to_select': [4, 5], 'model_regression__bagging_temperature': [0.0, 1.0], 'model_regression__depth': [4, 6, ...], 'model_regression__iterations': [100, 250, ...], ...}"
,n_iter,5
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [8]:
best_model_pipeline = random_search.best_estimator_

preprocessor_step = best_model_pipeline.named_steps['preprocessing']
sfs_step = best_model_pipeline.named_steps['feature_selection']

kolom_setelah_transformasi = preprocessor_step.get_feature_names_out()
fitur_terpilih = kolom_setelah_transformasi[sfs_step.get_support()]

print(f'\nHyperparameter Terbaik:\n{random_search.best_params_}')
print(f'\nFitur Terbaik Yang Terpilih:\n{list(fitur_terpilih)}')

#cek score
r2_score_train = best_model_pipeline.score(X_train, y_train)
r2_score_test = best_model_pipeline.score(X_test, y_test)

y_pred_test = best_model_pipeline.predict(X_test)
y_pred_train = best_model_pipeline.predict(X_train)

print(f'\nAkurasi Validasi Terbaik: {random_search.best_score_*100:.2f}%')



Hyperparameter Terbaik:
{'model_regression__random_strength': 1, 'model_regression__learning_rate': 0.05, 'model_regression__l2_leaf_reg': 3, 'model_regression__iterations': 100, 'model_regression__depth': 4, 'model_regression__bagging_temperature': 1.0, 'feature_selection__n_features_to_select': 5}

Fitur Terbaik Yang Terpilih:
['categori_encoding__ocean_proximity_INLAND', 'categori_encoding__ocean_proximity_ISLAND', 'categori_encoding__ocean_proximity_NEAR BAY', 'categori_encoding__ocean_proximity_NEAR OCEAN', 'remainder__median_income']

Akurasi Validasi Terbaik: 60.77%


In [9]:
def evaluate_model(pred,test,score_umum,evaluate_type,model_name='Simple LinearRegression'):
    r2 = r2_score(pred,test)
    mse = mean_squared_error(pred,test)
    mae = mean_absolute_error(pred,test)
    rmse = root_mean_squared_error(pred,test)
    mape = mean_absolute_percentage_error(pred,test)

    data = {
        'Evaluated Data': [evaluate_type],
        'Score Umum': [score_umum*100],
        'R2 Score': [r2],
        'MAE': [mae],
        'MSE': [mse],
        'RMSE': [rmse],
        'MAPE': [mape]
    }

    df_result = pd.DataFrame(data,index=[model_name])
    return df_result

In [10]:
def check_model_fit(df_eval_train,df_eval_test):
    df_combined = pd.concat([df_eval_train,df_eval_test],ignore_index=True)
    r2_train_score = df_eval_train['R2 Score'].values[0]
    r2_test_score = df_eval_test['R2 Score'].values[0]
    gap = r2_train_score -  r2_test_score

    if  r2_score_train < 0.60 or r2_test_score < 0.60:
        status = 'Underfitting (Performa Rendah pada Kedua Data)'
    elif gap > 0.05:
        status = f'Overfitting (Gap Terlalu Jauh: {gap*100:.2f}%)'
    elif gap < -0.05:
        status = f'Data Leakage / Bias (Test > Train, Gap: {gap*100:.2f}%)'
    else:
        status = 'Good Fit (Model Stabil)'

    df_combined['Status'] = status
    return df_combined

In [11]:
df_eval_train = evaluate_model(y_pred_train,y_train,score_umum=r2_score_train,evaluate_type='Training')
df_eval_test = evaluate_model(y_pred_test,y_test,score_umum=r2_score_test,evaluate_type='Test')
df_eval = check_model_fit(df_eval_train,df_eval_test)

print(f"{'=' * 50} PREDIKSI DENGAN SAMPLE DATASET {'=' * 68}")
print(df_eval.to_string())
print("\n" + "="* 150 + "\n")

================================================== PREDIKSI DENGAN SAMPLE DATASET ====================================================================
  Evaluated Data  Score Umum  R2 Score           MAE           MSE          RMSE      MAPE                                          Status
0       Training   60.968795  0.331943  52143.246783  5.217606e+09  72233.000795  0.267853  Underfitting (Performa Rendah pada Kedua Data)
1           Test   58.849946  0.307939  52398.473410  5.392340e+09  73432.554502  0.270743  Underfitting (Performa Rendah pada Kedua Data)




In [12]:
data = {
    "longitude": [-122.23, -122.22, -122.24],
    "latitude": [37.88, 37.86, 37.85],
    "housing_median_age": [41, 21, 52],
    "total_rooms": [880, 7099, 1467],
    "total_bedrooms": [129.0, 1106.0, 190.0],
    "population": [322, 2401, 496],
    "households": [126, 1138, 177],
    "median_income": [8.3252, 8.3014, 7.2574],
    "ocean_proximity": ["NEAR BAY", "NEAR BAY", "NEAR BAY"],
    "median_house_value": [452600, 358500, 352100],
}
df_new = pd.DataFrame(data)
data_rumahBaru_x = df_new.drop(columns='median_house_value')
data_rumahBaru_y = df_new['median_house_value']

In [13]:
print("=== Melakukan Prediksi Data Harga Rumah Baru ===")
prediksi_rumah = best_model_pipeline.predict(data_rumahBaru_x)

hasil_df = data_rumahBaru_x.copy()
hasil_df['Harga Prediksi'] = prediksi_rumah
hasil_df['Harga Asli'] = data_rumahBaru_y
hasil_df['Selisih Harga Asli VS Prediksi'] = hasil_df['Harga Prediksi'] - hasil_df['Harga Asli']

akurasi_bawaan = best_model_pipeline.score(data_rumahBaru_x,data_rumahBaru_y)
df_eval_data_baru = evaluate_model(
    pred=prediksi_rumah,
    test=data_rumahBaru_y,
    score_umum=akurasi_bawaan,
    evaluate_type='Data_Baru'
)
print(f"Akurasi Model: {akurasi_bawaan * 100:.2f}%")
print("\nTabel Skor Evaluasi Lengkap Data Baru:")
print(df_eval_data_baru.to_string(index=False))
hasil_df[['Harga Prediksi','Harga Asli','Selisih Harga Asli VS Prediksi']]

=== Melakukan Prediksi Data Harga Rumah Baru ===
Akurasi Model: -46.30%

Tabel Skor Evaluasi Lengkap Data Baru:
Evaluated Data  Score Umum  R2 Score          MAE          MSE         RMSE     MAPE
     Data_Baru  -46.301754 -6.291785 46838.577824 3.087946e+09 55569.286587 0.109762


,Harga Prediksi,Harga Asli,Selisih Harga Asli VS Prediksi
0,442169.745506,452600,-10430.254494
1,442169.745506,358500,83669.745506
2,398515.733471,352100,46415.733471
